# 01. 주행 영상에서 눈부심·터널 피처 뽑기

**한 줄 요약:** 영상을 사진(프레임) 묶음으로 보고, 각 사진의 **픽셀 숫자**로 밝기·눈부심·시야·움직임을 계산해 **표(CSV)** 로 만든다. 딥러닝 모델을 학습하지 않고 OpenCV 계산만 쓴다.

### 영상이 숫자가 되는 원리
- 영상 = 1초에 30장(30fps) 넘어가는 사진.
- 사진 1장 = 가로×세로 픽셀 격자, 픽셀마다 밝기 숫자 **0(검정) ~ 255(흰색)**.
- 해를 정면으로 보면 픽셀이 **255에 붙어버린다(포화)**. 이 비율이 곧 "눈부심 정도".
- 터널에 들어가면 평균 밝기가 **갑자기 뚝 떨어지고**, 나오면 **확 뛴다**. 이 변화 크기와 속도가 "적응 부담".

### 이 노트북 순서
1. 프레임 1장에서 피처 계산 (`frame_features`)
2. 영상 전체를 초당 몇 장씩 훑으면서 시계열 표 만들기 (`process_video`)
3. 밝기 시계열에서 터널 진입·탈출 순간 자동 찾기 (`detect_tunnel_events`)
4. 영상 1개 → 한 줄 요약 (`summarize_clip`)
5. 데모 영상으로 바로 실행해보기 → 실제 영상 폴더 일괄 처리

준비: 터미널에서 `pip install opencv-python numpy pandas matplotlib`

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"   # 윈도우 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

VIDEO_DIR = Path("videos")          # 실제 영상(mp4)을 넣을 폴더
OUT_DIR = Path("data/video")        # 결과 CSV 저장 폴더
VIDEO_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_FPS = 5      # 1초에 5장만 분석 (30장 다 보면 느리고 결과는 비슷)
RESIZE_W = 640      # 가로 640px로 줄여서 계산 (속도용)
SAT_LEVEL = 250     # 이 밝기 이상이면 '포화(하얗게 날아간)' 픽셀로 봄

## 1. 프레임 1장에서 뽑는 피처

| 피처 | 뜻 | 왜 필요한가 |
|---|---|---|
| `lum_mean` | 화면 평균 밝기 (0~255) | 터널 안/밖, 역광 여부 |
| `sat_ratio` | 화면 중 하얗게 날아간 픽셀 비율 | **눈부심 강도**의 핵심 지표 |
| `sat_upper` | 화면 위쪽 절반의 포화 비율 | 해·하늘에서 오는 직사광 눈부심 |
| `sun_blob` | 가장 큰 포화 덩어리 크기 비율 | 해가 화면 안에 직접 보이는지 |
| `sun_y` | 그 덩어리의 세로 위치 (0=맨 위, 1=맨 아래) | 해가 낮게 떠 있을수록 시야 정면 → 위험 |
| `contrast` | 화면 중앙의 밝기 대비 (표준편차) | 눈부시면 물체가 뿌옇게 보여 대비가 떨어짐 |
| `edge_density` | 도로 영역에서 윤곽선 비율 | 차선·앞차 윤곽이 보이는 정도 = **운전자 시야** 대용 |

> 운전자 시야를 직접 잴 수는 없어서, "카메라가 앞을 얼마나 또렷하게 보는가"를 `contrast`와 `edge_density`로 대신 잰다.

In [ ]:
def frame_features(frame_bgr):
    # 컬러 → 밝기(흑백) 한 장으로 변환
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    sat = gray >= SAT_LEVEL

    # 가장 큰 포화 덩어리 = 해(또는 강한 반사광) 후보
    n, labels, stats, cents = cv2.connectedComponentsWithStats(sat.astype(np.uint8), connectivity=8)
    if n > 1:
        i = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        sun_blob = stats[i, cv2.CC_STAT_AREA] / (h * w)
        sun_y = cents[i, 1] / h
    else:
        sun_blob, sun_y = 0.0, np.nan

    center = gray[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
    road = gray[h // 2:, w // 6: 5 * w // 6]           # 화면 아래쪽 가운데 = 도로
    edges = cv2.Canny(road, 50, 150)

    return {
        "lum_mean": float(gray.mean()),
        "sat_ratio": float(sat.mean()),
        "sat_upper": float(sat[: h // 2].mean()),
        "sun_blob": float(sun_blob),
        "sun_y": float(sun_y),
        "contrast": float(center.std()),
        "edge_density": float((edges > 0).mean()),
    }

## 2. 영상 전체 훑기 + 움직임(상대 속도)

**옵티컬 플로우(optical flow)**: 연속된 두 장을 비교해서 "각 픽셀이 몇 칸 이동했나"를 계산하는 OpenCV 기능.
차가 빠를수록 도로 위 차선·노면이 화면에서 빨리 흘러가므로 이동량이 커진다.

- 결과 `flow_speed`는 **초당 픽셀 이동량**이라 km/h가 아니라 **상대 속도 지표**다.
- 같은 카메라·같은 차 영상끼리 "빨라졌다/느려졌다" 비교에는 충분하다.
- km/h가 꼭 필요하면 BDD100K처럼 **GPS 속도가 같이 들어 있는 데이터**를 쓰면 된다.

In [ ]:
def process_video(path, sample_fps=SAMPLE_FPS, resize_w=RESIZE_W, max_sec=None):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError(f"영상을 열 수 없음: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    step = max(1, round(fps / sample_fps))

    rows, prev_small, idx = [], None, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            t = idx / fps
            if max_sec and t > max_sec:
                break
            h, w = frame.shape[:2]
            frame = cv2.resize(frame, (resize_w, int(h * resize_w / w)))
            feat = frame_features(frame)

            # 옵티컬 플로우는 더 작게 줄여서 계산 (빠르게)
            small = cv2.cvtColor(cv2.resize(frame, (resize_w // 2, frame.shape[0] // 2)), cv2.COLOR_BGR2GRAY)
            if prev_small is not None:
                flow = cv2.calcOpticalFlowFarneback(prev_small, small, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                mag = np.linalg.norm(flow, axis=2)
                feat["flow_speed"] = float(np.percentile(mag[mag.shape[0] // 2:], 90) * fps / step)  # 도로 영역 상위 10% 이동량, px/초
            else:
                feat["flow_speed"] = np.nan
            prev_small = small

            feat["t"] = round(t, 3)
            rows.append(feat)
        idx += 1
    cap.release()
    return pd.DataFrame(rows).set_index("t")

## 3. 터널 진입·탈출 순간 자동 찾기

아이디어: 매 순간 **직전 1초 평균 밝기**와 **직후 1초 평균 밝기**를 비교한다.
- 직후가 직전보다 `ratio_thr`배(기본 2.5배) 이상 **어두우면 → 진입**
- 2.5배 이상 **밝으면 → 탈출**

각 이벤트마다 남기는 값:
- `lum_ratio`: 밝기 몇 배 차이 (클수록 눈이 적응해야 할 양이 큼)
- `max_rate`: 초당 최대 밝기 변화량 (클수록 갑작스러움)
- `sat_after`: 탈출 직후 1초간 포화 비율 (터널 나오자마자 **하얗게 눈부신 정도**)
- `recover_s`: 밝기가 새 수준에 자리 잡기까지 걸린 시간 (카메라의 "적응 시간")

📚 근거: Mehri 외(2017)는 터널 입구 위험도를 `log(터널 밖 밝기 / 터널 안 밝기)`로 평가한다 → 우리 `lum_ratio`의 로그값이 같은 개념(이른바 **블랙홀 효과** 강도). 사람 눈 순응시간은 약 3초, 100km/h면 그동안 약 83m를 거의 못 보고 달린다(홍기혁).

⚠️ 블랙박스 카메라는 **자동 노출**이 있어서 1초 안팎이면 밝기를 스스로 맞춘다. 그래서 변화의 **크기와 첫 1초**를 보는 게 중요하다. (사람 눈은 카메라보다 훨씬 느리게 적응하므로, 카메라에서 크게 튀었다면 사람에겐 더 큰 부담이라고 해석)

In [ ]:
def detect_tunnel_events(df, ratio_thr=2.5, win_sec=1.0, min_gap_sec=3.0):
    lum = df["lum_mean"]
    dt = np.median(np.diff(df.index.values))
    k = max(1, int(round(win_sec / dt)))
    before = lum.rolling(k, min_periods=k).mean()                   # 직전 1초 평균
    after = lum[::-1].rolling(k, min_periods=k).mean()[::-1].shift(-1)  # 직후 1초 평균
    log_r = np.log((after + 1) / (before + 1))
    rate = lum.diff() / dt

    events, last_t = [], -1e9
    cand = log_r[np.abs(log_r) >= np.log(ratio_thr)]
    for t in cand.abs().sort_values(ascending=False).index:      # 변화 큰 순서로
        if any(abs(t - e["t"]) < min_gap_sec for e in events):   # 같은 이벤트 중복 제거
            continue
        kind = "exit" if log_r[t] > 0 else "entry"
        win = rate.loc[t - win_sec: t + win_sec]
        after_sat = df["sat_ratio"].loc[t: t + win_sec]
        # 회복 시간: 새 밝기 수준(2~4초 뒤 중앙값)의 ±10% 안으로 들어올 때까지 걸린 시간
        level = lum.loc[t + 2: t + 4].median()
        settled = lum.loc[t:][(lum.loc[t:] - level).abs() <= 0.1 * level]
        recover_s = float(settled.index[0] - t) if len(settled) else np.nan
        events.append({
            "t": t, "type": kind,
            "lum_before": round(before[t], 1), "lum_after": round(after[t], 1),
            "lum_ratio": round(float(np.exp(abs(log_r[t]))), 2),
            "max_rate": round(float(win.abs().max()), 1),
            "sat_after": round(float(after_sat.mean()), 4),
            "recover_s": recover_s,
        })
    return pd.DataFrame(events).sort_values("t").reset_index(drop=True) if events else pd.DataFrame(
        columns=["t", "type", "lum_before", "lum_after", "lum_ratio", "max_rate", "sat_after", "recover_s"])

## 4. 영상 1개 → 요약 한 줄

ML 모델은 "행 = 관측 1개" 표를 받는다. 영상 하나를 대표값으로 줄인다.
- 눈부심: 평균·최대 포화 비율, 해가 보인 시간 비율
- 시야: 대비·윤곽선이 가장 나빴던 수준 (하위 10%)
- 터널: 진입/탈출 횟수, 가장 컸던 밝기 비율, 탈출 직후 포화

In [ ]:
def summarize_clip(df, events, name=""):
    ev = events if len(events) else pd.DataFrame(columns=["type", "lum_ratio", "max_rate", "sat_after"])
    ex = ev[ev["type"] == "exit"]
    return {
        "clip": name,
        "duration_s": float(df.index.max()),
        "lum_mean": df["lum_mean"].mean(),
        "sat_mean": df["sat_ratio"].mean(),
        "sat_max": df["sat_ratio"].max(),
        "sun_visible_frac": (df["sun_blob"] > 0.002).mean(),
        "contrast_p10": df["contrast"].quantile(0.1),
        "edge_p10": df["edge_density"].quantile(0.1),
        "flow_speed_med": df["flow_speed"].median(),
        "n_entry": int((ev["type"] == "entry").sum()),
        "n_exit": int(len(ex)),
        "tunnel_lum_ratio_max": ev["lum_ratio"].max() if len(ev) else np.nan,
        "tunnel_max_rate": ev["max_rate"].max() if len(ev) else np.nan,
        "exit_sat_after_max": ex["sat_after"].max() if len(ex) else np.nan,
        "recover_s_max": ev["recover_s"].max() if len(ev) else np.nan,
    }

## 5. 데모: 가짜 터널 영상으로 바로 돌려보기

실제 영상이 없어도 동작을 확인할 수 있게, 코드로 20초짜리 영상을 만든다.
- 0~6초: 맑은 바깥 / 6~13초: 터널 안(어두움) / 13초~: 터널 탈출 + 정면에 해(눈부심)
- 차선이 아래로 흘러가서 옵티컬 플로우가 잡힌다.

In [ ]:
def make_demo_video(path, sec=20, fps=30, w=640, h=360):
    out = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for i in range(sec * fps):
        t = i / fps
        in_tunnel = 6 <= t < 13
        sky, road = (60, 50, 45) if in_tunnel else (235, 200, 170), (35, 35, 35) if in_tunnel else (110, 110, 110)
        img = np.zeros((h, w, 3), np.uint8)
        img[: h // 2] = sky
        img[h // 2:] = road
        lane = 255 if not in_tunnel else 120
        for y0 in range(-60, h // 2, 60):                       # 아래로 흐르는 차선
            y = h // 2 + (y0 + int(t * 240)) % (h // 2)
            cv2.rectangle(img, (w // 2 - 4, y), (w // 2 + 4, min(h - 1, y + 25)), (lane,) * 3, -1)
        if t >= 13:                                              # 탈출 후 정면의 해
            r = int(25 + 15 * min(1, (t - 13) / 3))
            cv2.circle(img, (w // 2 + 40, h // 2 - 30), r * 3, (255, 255, 255), -1)
            img = cv2.GaussianBlur(img, (0, 0), 6)
            cv2.circle(img, (w // 2 + 40, h // 2 - 30), r * 2, (255, 255, 255), -1)
        noise = np.random.randint(0, 8, img.shape, dtype=np.uint8)
        out.write(cv2.add(img, noise))
    out.release()

demo_path = VIDEO_DIR / "_demo_tunnel.mp4"
make_demo_video(demo_path)
demo_df = process_video(demo_path)
demo_events = detect_tunnel_events(demo_df)
demo_events

In [ ]:
def plot_clip(df, events, title=""):
    fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
    df["lum_mean"].plot(ax=axes[0], title="평균 밝기 (0~255)")
    df["sat_ratio"].plot(ax=axes[1], color="orange", title="포화 픽셀 비율 = 눈부심")
    df[["contrast"]].plot(ax=axes[2], color="green", title="대비 (낮을수록 뿌옇게 보임)", legend=False)
    df["flow_speed"].plot(ax=axes[3], color="purple", title="흐름 속도 (상대 속도, px/초)")
    for _, e in events.iterrows():
        for ax in axes:
            ax.axvline(e["t"], color="red" if e["type"] == "exit" else "blue", ls="--", lw=1)
    axes[-1].set_xlabel("시간 (초)  ·  파란 점선=터널 진입, 빨간 점선=탈출")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

plot_clip(demo_df, demo_events, "데모 영상")
pd.Series(summarize_clip(demo_df, demo_events, demo_path.name))

## 6. 실제 영상 일괄 처리

1. `videos/` 폴더에 mp4 파일들을 넣는다 (파일명 앞에 `_`가 붙은 데모는 건너뜀).
2. 아래 셀을 실행하면
   - `data/video/frames/<영상이름>.csv` : 영상별 초 단위 시계열
   - `data/video/clip_features.csv` : 영상 1개 = 1행 요약표 ← **ML에 들어갈 표**

In [ ]:
frames_dir = OUT_DIR / "frames"
frames_dir.mkdir(exist_ok=True)

summaries, all_events = [], []
videos = sorted(p for p in VIDEO_DIR.glob("*.mp4") if not p.name.startswith("_"))
print(f"영상 {len(videos)}개 처리 시작")
for p in videos:
    try:
        df = process_video(p)
    except IOError as e:
        print("건너뜀:", e)
        continue
    ev = detect_tunnel_events(df)
    df.to_csv(frames_dir / f"{p.stem}.csv")
    summaries.append(summarize_clip(df, ev, p.name))
    all_events.append(ev.assign(clip=p.name))
    print(f"  {p.name}: {len(df)}프레임, 터널 이벤트 {len(ev)}개")

if summaries:
    clip_df = pd.DataFrame(summaries)
    clip_df.to_csv(OUT_DIR / "clip_features.csv", index=False, encoding="utf-8-sig")
    pd.concat(all_events).to_csv(OUT_DIR / "tunnel_events.csv", index=False, encoding="utf-8-sig")
    display(clip_df)

## 다음 단계: 이 표가 사고확률 모델에 들어가는 방식

영상은 사고 기록(도로공사 4,458건)과 **1:1로 짝지어지지 않는다**. 그래서 두 가지 방식으로 쓴다.

1. **눈부심 보정 모델 (메인)**
   영상마다 촬영 시각·위치·진행방향을 알면 태양 고도·방위각을 계산할 수 있다(pvlib).
   → `X = 태양 고도, 태양-진행방향 각도차, 구름량` / `y = 영상에서 잰 sat_ratio`
   로 작은 회귀 모델을 학습하면 "이 각도면 눈부심이 이만큼"이라는 **눈부심 지수**를 얻는다.
   이 지수를 사고 데이터의 모든 구간×시간에 계산해서 **피처로 넣는다**. (논문의 ±57.7° 같은 고정 기준 대신, 실측으로 보정한 연속값)

2. **터널 적응 부담 지수**
   `tunnel_events.csv`의 `lum_ratio`, `sat_after`를 시간대·방향별로 평균 내면 "아침 동향 출구는 밝기 차 4배" 같은 값이 나온다. 터널 데이터의 출구 좌표·방향과 붙여 터널별 피처로 쓴다.

3. **보조 검증 (Nexar 데이터)**
   사고 영상 vs 일반 영상에서 `sat_mean`, `contrast_p10`, `exit_sat_after_max` 차이를 로지스틱 회귀/랜덤포레스트로 비교 → "눈부심·밝기 급변이 사고 영상에 더 많다"는 근거 자료.